In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 06 · MCP servers are OAuth 2.1 resource servers

**Primer section:** §7.1 MCP — the server is an OAuth 2.1 resource server.

The MCP authorization spec fixes the roles: the MCP client is an OAuth 2.1 client, the MCP server is
a **resource server**, and an authorization server issues tokens. The requirements you should be
able to recite — Protected Resource Metadata (RFC 9728), PKCE, the `resource` parameter (RFC 8707),
audience validation, **no token passthrough**, `403 insufficient_scope` — are all implemented by the
tickets server in `agentsec.mcp.server` on top of the `mcp` SDK's native auth. This notebook talks to
it raw over JSON-RPC first, then through an ADK `McpToolset` with per-request delegated tokens.

In [ ]:
import logging

from agentsec.logging_utils import quiet_logs

quiet_logs(logging.ERROR)

import json

import httpx
import jwt  # display only
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

from agentsec.agents import LocalStack, Step, build_support_agent
from agentsec.audit import AuditLog
from agentsec.config import Settings
from agentsec.identity import DPoP, TokenIssuer, public_jwk
from agentsec.mcp import (
    SCOPE_READ,
    SCOPE_WRITE,
    TICKETS,
    ServerThread,
    build_server,
    delegated_token_minter,
    free_port,
    make_mcp_toolset,
)
from agentsec.runtime import run_turn, seed_session

issuer = TokenIssuer()      # the authorization server every party trusts
audit = AuditLog()

port = free_port()
url = f"http://127.0.0.1:{port}/mcp"          # the server's canonical URI = the RFC 8707 resource = token `aud`
srv = build_server(issuer, resource_url=url, audit=audit, require_dpop=False)
server = ServerThread(srv.app(require_dpop=False), port=port).start()
print("tickets MCP server listening at", url)

HEADERS = {
    "Accept": "application/json, text/event-stream",
    "Content-Type": "application/json",
    "MCP-Protocol-Version": "2025-11-25",
}

def rpc(target_url: str, token: str | None = None, *, method: str = "tools/list", params: dict | None = None, headers: dict | None = None) -> httpx.Response:
    h = dict(HEADERS)
    if token:
        h["Authorization"] = f"Bearer {token}"
    h.update(headers or {})
    body = {"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}}
    return httpx.post(target_url, headers=h, json=body, timeout=10)

def call_tool(target_url: str, token: str | None, name: str, **arguments):
    return rpc(target_url, token, method="tools/call", params={"name": name, "arguments": arguments})

def peek(token: str) -> dict:
    return jwt.decode(token, options={"verify_signature": False})

## 1. Discovery: Protected Resource Metadata and the 401 challenge

A client that hits the server without a token gets `401` with
`WWW-Authenticate: Bearer resource_metadata="…"`; the PRM document names the resource's canonical
URI, its authorization server(s), and the scopes it supports. From there the client fetches the AS
metadata (RFC 8414 — `issuer.metadata()` here), sees `code_challenge_methods_supported: [S256]`, and
runs the PKCE flow with `resource=<canonical URI>`.

In [ ]:
r = rpc(url)  # no token
print("status:", r.status_code)
print("WWW-Authenticate:", r.headers["www-authenticate"])
assert r.status_code == 401 and 'resource_metadata="' in r.headers["www-authenticate"]

base = url.rsplit("/mcp", 1)[0]
prm = httpx.get(f"{base}/.well-known/oauth-protected-resource/mcp", timeout=5).json()
print("\nPRM:", json.dumps(prm, indent=2))
assert prm["resource"].rstrip("/") == url and prm["authorization_servers"][0].rstrip("/") == issuer.issuer

as_meta = issuer.metadata()
print("\nAS metadata subset:", {k: as_meta[k] for k in ("issuer", "token_endpoint", "code_challenge_methods_supported", "grant_types_supported")})

## 2. Audience validation and scope challenges

The same AS signs tokens for many resources. A token minted for `https://other.example/mcp` is
validly signed — and rejected here (`401`), because `aud` is not this server. A token for this
server without the required scope gets `403 insufficient_scope` with a `scope` challenge, which is
what drives step-up authorization.

In [ ]:
wrong_aud = issuer.mint(subject="u-ana", audience="https://other.example/mcp", scope=SCOPE_READ)
r = rpc(url, wrong_aud)
print("wrong audience →", r.status_code, "|", r.headers["www-authenticate"].split(",")[0])
assert r.status_code == 401

no_scope = issuer.mint(subject="u-ana", audience=url, scope="")
r = rpc(url, no_scope)
print("no scope      →", r.status_code, "|", r.headers["www-authenticate"].split(", resource_metadata")[0])
assert r.status_code == 403 and "insufficient_scope" in r.headers["www-authenticate"]

read_token = issuer.mint(subject="u-ana", audience=url, scope=SCOPE_READ)
r = rpc(url, read_token)
print("read token    →", r.status_code)
assert r.status_code == 200
print("\nserver-side audit of the rejection:", [(e.decision, e.reasons) for e in audit.events(lambda e: e.event_type == "mcp.auth")][-1])

## 3. `tools/list` annotations and scope → tool mapping

Tool annotations (`readOnlyHint`, `destructiveHint`) tell clients and gateways the tier of each tool —
VPC-SC's `mcp.tool.isReadOnly` condition keys off the same idea. Scopes map to tools: `tickets:read`
for the read-only tools, `tickets:write` for `refund_ticket`. A read token can list and get, but a
call to `refund_ticket` is a tool error, not a refund.

In [ ]:
tools = {t["name"]: t for t in rpc(url, read_token).json()["result"]["tools"]}
for name, t in tools.items():
    print(f"{name:<16} {t['annotations']}")
assert tools["get_ticket"]["annotations"]["readOnlyHint"] is True
assert tools["refund_ticket"]["annotations"]["destructiveHint"] is True

got = call_tool(url, read_token, "get_ticket", ticket_id="T-1").json()["result"]
print("\nget_ticket with read scope   →", got["structuredContent"])

denied = call_tool(url, read_token, "refund_ticket", ticket_id="T-1", amount=10).json()["result"]
print("refund_ticket with read scope →", "isError:", denied["isError"], "|", denied["content"][0]["text"])
assert denied["isError"] is True and "insufficient_scope" in denied["content"][0]["text"]

# Scope is necessary but not sufficient: a write-scoped token under the agent's OWN authority cannot refund either.
own_write = issuer.mint(subject="spiffe://agents.global.org-1.system.id.goog/resources/aiplatform/projects/1/locations/l/reasoningEngines/a", audience=url, scope=f"{SCOPE_READ} {SCOPE_WRITE}")
own = call_tool(url, own_write, "refund_ticket", ticket_id="T-1", amount=10).json()["result"]
print("refund_ticket, own authority  →", own["structuredContent"])
assert own["structuredContent"]["error"] == "forbidden"

## 4. A DPoP-required server: bearer rejected, proof accepted, replay rejected

With `require_dpop=True` the server insists on `Authorization: DPoP <token>` plus a `DPoP` proof
whose key thumbprint matches the token's `cnf.jkt`. The bound token presented as a plain Bearer is
refused; a replayed proof (same `jti`) is refused; a proof signed by a different key is refused.
This is what Agent Gateway enforces on Google Cloud.

In [ ]:
dpop_port = free_port()
dpop_url = f"http://127.0.0.1:{dpop_port}/mcp"
dpop_srv = build_server(issuer, resource_url=dpop_url, audit=AuditLog(), require_dpop=True)
dpop_server = ServerThread(dpop_srv.app(require_dpop=True), port=dpop_port).start()

key = DPoP.generate_key()
bound = issuer.mint_dpop_bound_token(subject="u-ana", audience=dpop_url, dpop_public_jwk=public_jwk(key), scope=SCOPE_READ)

r = rpc(dpop_url, bound)
print("bound token as Bearer            →", r.status_code, r.headers.get("www-authenticate", "")[:60])
assert r.status_code == 401

proof = DPoP.proof(key, method="POST", url=dpop_url, access_token=bound)
r = rpc(dpop_url, headers={"Authorization": f"DPoP {bound}", "DPoP": proof})
print("DPoP token + fresh proof         →", r.status_code)
assert r.status_code == 200

r = rpc(dpop_url, headers={"Authorization": f"DPoP {bound}", "DPoP": proof})
print("same proof replayed              →", r.status_code, "|", r.headers["www-authenticate"][:80])
assert r.status_code == 401 and "already used" in r.headers["www-authenticate"]

other_proof = DPoP.proof(DPoP.generate_key(), method="POST", url=dpop_url, access_token=bound)
r = rpc(dpop_url, headers={"Authorization": f"DPoP {bound}", "DPoP": other_proof})
print("proof from a different key       →", r.status_code)
assert r.status_code == 401

dpop_server.stop()

## 5. ADK `McpToolset` with a delegated, audience-bound token per request

`delegated_token_minter` turns the session's authority (the front-end-verified user) into a token
for **this server's audience** by RFC 8693 exchange — never by forwarding the user's token. The
`SecurityPlugin` still evaluates every `tickets_*` call against the policy first (tier, scopes,
envelope); the server then enforces audience, scope and row-level ownership independently.

In [ ]:
stack = LocalStack.create(Settings(mcp_url=url, sts_issuer=issuer.issuer))
stack.issuer = issuer  # share the AS with the server

minter = delegated_token_minter(issuer, agent=stack.agent_id, audience=url, scopes=[SCOPE_READ, SCOPE_WRITE])
sent_tokens: list[str] = []                                    # record what the toolset actually sends
def recording_minter(ctx):
    token = minter(ctx)
    sent_tokens.append(token)
    return token
toolset = make_mcp_toolset(url, token_minter=recording_minter)  # tool names get the prefix "tickets_"
agent = build_support_agent(model=stack.llm, extra_tools=[toolset])
runner = Runner(app_name="mcp-lab", agent=agent, plugins=[stack.plugin], session_service=InMemorySessionService())

await seed_session(runner, user_id="u-ana", session_id="s1",
                   user={"subject": "u-ana", "email": "ana@customer.example"}, scopes=[SCOPE_READ, SCOPE_WRITE])
stack.script(
    Step.call("tickets_list_tickets"),
    Step.call("tickets_get_ticket", ticket_id="T-3"),                  # Ben's ticket
    Step.call("tickets_refund_ticket", ticket_id="T-2", amount=35.0),  # Ana's, inside the envelope (≤ 50)
    Step.say("Listed your tickets and refunded T-2."),
)
r = await run_turn(runner, user_id="u-ana", session_id="s1", message="show my tickets and refund the museum pass")

by_name = {t["name"]: t["response"] for t in r.tool_responses}
listed = by_name["tickets_list_tickets"]["structuredContent"]["tickets"]
print("list_tickets (delegated: only Ana's rows):", [t["id"] for t in listed])
print("get_ticket T-3 (Ben's)                  :", by_name["tickets_get_ticket"]["structuredContent"])
print("refund_ticket T-2                       :", by_name["tickets_refund_ticket"]["structuredContent"])
print("fenced text block                       :", by_name["tickets_list_tickets"]["content"][0]["text"][:70], "…")

assert {t["id"] for t in listed} == {"T-1", "T-2"}
assert by_name["tickets_get_ticket"]["structuredContent"]["error"] == "forbidden"
assert by_name["tickets_refund_ticket"]["structuredContent"]["ticket_id"] == "T-2" and TICKETS["T-2"]["status"] == "refunded"

### No token passthrough: the upstream call used a *separate* token

`refund_ticket` calls a payments API. The MCP server is an OAuth client to that API: it obtains a token
for the payments audience under its **own** identity (`sub` = the server's SPIFFE ID), recording the
calling agent as `act` and the user as `on_behalf_of` — a fresh delegation hop. The token it received
from the MCP client — `aud` = the MCP server — never leaves the process.

In [ ]:
upstream = srv.payments.calls[-1]
claims = peek(upstream["token"])
print(f"upstream token: aud={claims['aud']} sub={claims['sub'].rsplit('/', 1)[-1]} scope={claims['scope']} on_behalf_of={claims['on_behalf_of']}")
assert claims["aud"] == "https://payments.acme.example" and claims["aud"] != url
assert claims["sub"] == srv.payments.server_identity  # the MCP server's OWN identity
assert claims["act"]["sub"] == stack.agent_id.spiffe_id and claims["on_behalf_of"] == "u-ana"

# And the token the *client* actually sent was for this server only, delegated from Ana with the agent as actor:
inbound = peek(sent_tokens[-1])
print(f"inbound token : aud={inbound['aud']} sub={inbound['sub']} act={inbound['act']['sub'].rsplit('/', 1)[-1]} scope={inbound['scope']}")
assert inbound["aud"] == url and inbound["act"]["sub"] == stack.agent_id.spiffe_id

await toolset.close()

## 6. Shut down and read the server's own audit trail

The server records auth failures and every tool decision with both identities when the token is
delegated — the same dual-identity shape as the runtime plugin.

In [ ]:
server.stop()
for e in audit.events(lambda e: e.event_type.startswith("mcp.")):
    print(f"{e.event_type:<10} {e.decision:<6} {e.tool or '-':<14} user={e.user or '-':<6} agent={e.agent.rsplit('/', 1)[-1]:<14} {e.reasons or ''}")

## On Google Cloud

* Host the server on Cloud Run with `--functional-type=mcp-server` (identity: agent identity or a
  service account); it is registered in **Agent Registry** and reached through **Agent Gateway**,
  where IAP enforces IAM on the registered MCP server resource and VPC-SC can condition on
  `mcp.toolName` / `mcp.method` / `mcp.tool.isReadOnly` (`infra/scripts/vpc_sc_mcp_rule.yaml`).
* For Google-hosted MCP endpoints (e.g. BigQuery), `McpToolset(auth_scheme=GcpAuthProviderScheme(…))`
  lets Auth Manager broker the user's 3-legged token per call.
* The 2026-07-28 spec revision keeps this model and deprecates Dynamic Client Registration in favour of
  Client ID Metadata Documents (`client_id_metadata_document_supported` in the AS metadata above).

**In one sentence:** "An MCP server is just an OAuth 2.1 resource server: it publishes
protected-resource metadata, the client uses PKCE and the `resource` parameter, the server validates
audience and scope, maps scopes to tools with read-only/destructive annotations, and never passes the
inbound token upstream — it gets its own. Sender-constrain with DPoP through the gateway so a
stolen token cannot be replayed."